# Notebook 6b: DeepJDOT (optimal transport inside the training loop)

The second optimal transport method and the last method of our study (Damodaran et al., 2018).

### How it differs from JDOT (notebook 6a)
* **JDOT** worked on fixed features: it computed *one* transport plan, read off soft labels, retrained a classifier, and repeated.
* **DeepJDOT** puts the transport problem **inside the training of the network**. At **every training step** it takes a small
  batch of lab images and a small batch of field images, finds the cheapest transport plan *between those two batches*, and adds
  the **cost of that plan to the loss**. Minimising the loss then reshapes the network so that matching lab and field images
  becomes cheap.

### The loss
**loss = classification loss (lab labels) + weight x transport cost**

The transport cost of a plan `gamma` between a lab batch and a field batch is the sum over all pairs (i, j) of
`gamma[i, j] x cost[i, j]`, where the cost has two parts, as in JDOT:
1. **feature distance** between lab image *i* and field image *j* (in the network's bottleneck features);
2. **label disagreement**: `-log` of the probability the network gives, for field image *j*, to lab image *i*'s label.

At each step the plan is **computed first and then held fixed** (it is not differentiated); the loss then pushes the network to
make the *costs of the pairs the plan chose* smaller.

### Our setting
Same skeleton as notebook 5 (Deep CORAL and JMMD): frozen 2048-number features, a bottleneck network (2048 to 256 to 21 classes),
the field batch drawn from each split's ~200 **calibration** images **without labels**, the same weight-selection rule
and the same search budget (five weights). The matched control is again **`bottleneck_only`** from notebook 5.

### Needs
`pip install POT` (for the transport step). A GPU is optional (the network is small); the transport step runs on the CPU.

### Plan
1. Settings
2. Load the features
3. The DeepJDOT loss, with a toy check
4. The model and the training function
5. Choose the weight
6. All 10 splits, then save
7. Evaluate with notebook 3b

## Step 1: Settings

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import ot                      # POT: Python Optimal Transport
except ImportError:
    raise ImportError("Install POT first, in a terminal:  pip install POT")

In [2]:
# ---- Edit these -------------------------------------------------------------
QUICK_TEST = False                 # True = 2 splits, 2 epochs, a small weight sweep. False = the real run.

METHOD = "deepjdot"
SOURCE_METHOD = "source_only"
TRAIN_SEED = 42

OUT_DIR = Path("outputs")
MANIFEST = OUT_DIR / "master_manifest.csv"
SEEDS = list(range(42, 52))       # the 10 Setting B splits
DEV_SEED = 42                     # the split used to choose the weight

# Same training settings as notebook 5, so the comparison is fair
BOTTLENECK = 256
BATCH_SIZE = 128
EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
TOLERANCE = 0.01                  # the chosen weight may cost at most this much lab-validation accuracy
SCALE_FLOOR = 0.5                 # ... and must keep the bottleneck features at least this fraction of their no-alignment size
EPS = 1e-6

# DeepJDOT-specific choices (the paper's values are for other datasets)
ALPHA = 1.0                       # weight of the feature-distance part of the cost
LABEL_WEIGHT = 1.0                # weight of the label-disagreement part of the cost
WEIGHT_GRID = [0.1, 0.3, 1, 3, 10]   # weights for the whole transport cost (five values, like the other methods)
# -----------------------------------------------------------------------------

weight_grid = WEIGHT_GRID
if QUICK_TEST:
    EPOCHS, weight_grid = 2, weight_grid[1:3]
seeds_to_run = SEEDS[:2] if QUICK_TEST else SEEDS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Method: {METHOD} | device: {device} | splits: {seeds_to_run} | epochs: {EPOCHS} | weights to try: {weight_grid}")
if QUICK_TEST:
    print("*** QUICK TEST: 2 splits, 2 epochs, 2 weights. Set QUICK_TEST = False for the real run. ***")

Method: deepjdot | device: cuda | splits: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51] | epochs: 20 | weights to try: [0.1, 0.3, 1, 3, 10]


## Step 2: Load the features

Identical to notebooks 4c and 5: PlantVillage training features (02b), validation features and PlantDoc features (02),
all standardised with the PlantVillage training statistics.

In [3]:
source = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(source["classes"])
n_classes = len(classes)

train = np.load(OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
train_features, train_labels = train["features"].astype(np.float32), train["labels"]

val_features, val_labels = source["pv_val_features"].astype(np.float32), source["pv_val_labels"]
known_features = source["pd_known_features"].astype(np.float32)
unknown_features = source["pd_unknown_features"].astype(np.float32)
field_labels = source["pd_known_labels"]
known_paths = source["pd_known_paths"]

mean, std = train_features.mean(axis=0), train_features.std(axis=0)
to_tensor = lambda x: torch.from_numpy((x - mean) / (std + EPS)).to(device)

src_x, val_x = to_tensor(train_features), to_tensor(val_features)
known_x, unknown_x = to_tensor(known_features), to_tensor(unknown_features)
src_y = torch.from_numpy(train_labels).long().to(device)
val_y = torch.from_numpy(val_labels).long().to(device)

manifest = pd.read_csv(MANIFEST, keep_default_na=False)
split_columns = [f"B_seed{s}" for s in SEEDS]
known_rows = manifest[(manifest["dataset"] == "pd") & (manifest["role"] == "pd_known")].set_index("path")
split_info = known_rows.loc[known_paths, split_columns].reset_index(drop=True)

print("source train:", tuple(src_x.shape), "| source validation:", tuple(val_x.shape),
      "| field known:", tuple(known_x.shape), "| field unknown:", tuple(unknown_x.shape))

source train: (18601, 2048) | source validation: (3986, 2048) | field known: (2018, 2048) | field unknown: (466, 2048)


## Step 3: The DeepJDOT loss

For one lab batch (features and labels) and one field batch (features and the network's predictions) we need:

**1. The cost table.** `cost[i, j]` = `ALPHA` x (feature distance between lab image *i* and field image *j*, divided by the
number of features) + `LABEL_WEIGHT` x (`-log` of the probability the network gives field image *j* for lab image *i*'s label).

**2. The plan.** `ot.emd` finds the cheapest way to move the whole lab batch onto the field batch. It runs on the CPU with
plain numbers, and it is **not** differentiated.

**3. The loss.** The sum of `plan x cost` over all pairs. The plan is fixed, but the **cost table** is computed from the network's
outputs, so the gradient flows through it and trains the network.

In [4]:
def transport_cost(feat_s, ys, logits_t, feat_t):
    d = feat_s.shape[1]
    feature_cost = (feat_s.unsqueeze(1) - feat_t.unsqueeze(0)).pow(2).sum(dim=2) / d          # (lab images x field images)
    label_cost = -F.log_softmax(logits_t, dim=1)[:, ys].t()                                    # (lab images x field images)
    return ALPHA * feature_cost + LABEL_WEIGHT * label_cost


def deepjdot_loss(feat_s, ys, logits_t, feat_t):
    cost = transport_cost(feat_s, ys, logits_t, feat_t)
    n_lab, n_field = cost.shape
    with torch.no_grad():                                                                       # the plan is not differentiated
        plan = ot.emd(np.full(n_lab, 1.0 / n_lab), np.full(n_field, 1.0 / n_field),
                      cost.detach().cpu().numpy().astype(np.float64))
    plan = torch.from_numpy(plan).float().to(cost.device)
    return (plan * cost).sum()

**Check it on a toy batch.** Eight lab points in two classes, and eight field points that are slightly shifted copies of them.
If the network predicts the field points' classes **correctly**, the transport cost should be low. If it predicts them
**wrongly** (the other class), the label part of the cost jumps.

In [5]:
torch.manual_seed(0)
labels = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
lab_feat = torch.randn(8, 4) + labels.unsqueeze(1).float() * 3.0          # two well separated classes
field_feat = lab_feat + 0.1 * torch.randn(8, 4)                            # field points: slightly shifted copies

def predictions(correct):
    wanted = labels if correct else 1 - labels
    return torch.nn.functional.one_hot(wanted, 21).float() * 8.0            # confident logits for the chosen class

print("network predicts field classes correctly:", round(float(deepjdot_loss(lab_feat, labels, predictions(True), field_feat)), 3))
print("network predicts field classes wrongly:  ", round(float(deepjdot_loss(lab_feat, labels, predictions(False), field_feat)), 3),
      "(expect a much larger number)")

network predicts field classes correctly: 0.018
network predicts field classes wrongly:   7.631 (expect a much larger number)


## Step 4: The model and the training function

Exactly the model and loop of notebook 5, with the DeepJDOT loss in place of CORAL/JMMD. The transport weight is ramped up
smoothly from 0, and training stops on lab-validation accuracy. At weight 0 the transport step is skipped, because it
would add nothing.

One thing to keep in mind: each plan is computed from only **128 lab images and 128 field images** drawn from ~200 field
images, with 21 classes. A batch rarely contains every class, so a single plan is noisy. That is a known limitation of the
mini-batch approach.

In [6]:
class AlignNet(nn.Module):
    def __init__(self, in_dim, n_classes, bottleneck=BOTTLENECK):
        super().__init__()
        self.bottleneck = nn.Sequential(nn.Linear(in_dim, bottleneck), nn.ReLU())
        self.classifier = nn.Linear(bottleneck, n_classes)

    def forward(self, x):
        feat = self.bottleneck(x)
        return self.classifier(feat), feat

In [7]:
def train_deepjdot(target_x, seed, weight, verbose=False):
    torch.manual_seed(seed)
    net = AlignNet(src_x.shape[1], n_classes).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    steps_per_epoch = len(src_x) // BATCH_SIZE
    total_steps = EPOCHS * steps_per_epoch
    best_acc, best_state, bad_epochs, last_transport = -1.0, None, 0, float("nan")

    for epoch in range(EPOCHS):
        net.train()
        order = torch.randperm(len(src_x), device=device)

        for step in range(steps_per_epoch):
            progress = (epoch * steps_per_epoch + step) / total_steps
            ramp = 2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0           # 0 at the start, close to 1 at the end

            idx = order[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
            xs, ys = src_x[idx], src_y[idx]
            logits_s, feat_s = net(xs)
            classification_loss = F.cross_entropy(logits_s, ys)
            loss = classification_loss

            if weight > 0:
                xt = target_x[torch.randint(len(target_x), (BATCH_SIZE,), device=device)]   # unlabelled field batch
                logits_t, feat_t = net(xt)
                transport = deepjdot_loss(feat_s, ys, logits_t, feat_t)
                loss = loss + weight * ramp * transport
                last_transport = float(transport)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        net.eval()
        with torch.no_grad():
            val_acc = float((net(val_x)[0].argmax(dim=1) == val_y).float().mean())
        if verbose:
            print(f"epoch {epoch + 1:2d} | class loss {classification_loss.item():.3f} | transport cost {last_transport:.3f} | "
                  f"lab validation accuracy {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc, bad_epochs = val_acc, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        feature_size = float(net(val_x)[1].norm(dim=1).mean())            # average length of the bottleneck feature vector
    return net, best_acc, last_transport, feature_size


@torch.no_grad()
def logits_of(net, x):
    return net(x)[0].cpu().numpy().astype(np.float32)

## Step 5: Choose the weight

The same rule as notebook 5, with no field labels: train on the development split with **weight 0** (no transport penalty) and
with each candidate weight, then keep the **largest weight that (a) costs at most 1 point of lab-validation accuracy and
(b) keeps the bottleneck features at least half their no-penalty size**. The second condition stops the network from
"winning" by shrinking its features.

In [8]:
dev_cal = (split_info[f"B_seed{DEV_SEED}"] == "cal").to_numpy()
dev_target = known_x[torch.from_numpy(dev_cal).to(device)]
print(f"Development split {DEV_SEED}: {len(dev_target)} unlabelled field images")

rows = []
for weight in [0] + list(weight_grid):
    start = time.time()
    net, val_acc, transport_value, feature_size = train_deepjdot(dev_target, DEV_SEED, weight)
    with torch.no_grad():
        probs = F.softmax(net(dev_target)[0], dim=1)
        entropy = float(-(probs * torch.log(probs.clamp_min(1e-12))).sum(dim=1).mean())
    rows.append({"weight": weight, "lab validation accuracy": round(val_acc, 4),
                 "final transport cost": round(transport_value, 4), "feature size": round(feature_size, 3),
                 "field entropy (info only)": round(entropy, 3), "seconds": round(time.time() - start)})
    print(rows[-1])

sweep = pd.DataFrame(rows).set_index("weight")
sweep

Development split 42: 202 unlabelled field images


C:\Users\MIKE\AppData\Local\Temp\ipykernel_1752\3859356677.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  dev_target = known_x[torch.from_numpy(dev_cal).to(device)]


{'weight': 0, 'lab validation accuracy': 0.9977, 'final transport cost': nan, 'feature size': 51.718, 'field entropy (info only)': 0.359, 'seconds': 4}


C:\Users\MIKE\AppData\Local\Temp\ipykernel_1752\1615358034.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  last_transport = float(transport)


{'weight': 0.1, 'lab validation accuracy': 0.9955, 'final transport cost': 1.355, 'feature size': 26.954, 'field entropy (info only)': 0.995, 'seconds': 4}
{'weight': 0.3, 'lab validation accuracy': 0.9967, 'final transport cost': 1.0251, 'feature size': 12.082, 'field entropy (info only)': 0.907, 'seconds': 12}
{'weight': 1, 'lab validation accuracy': 0.9967, 'final transport cost': 0.9421, 'feature size': 13.286, 'field entropy (info only)': 0.862, 'seconds': 8}
{'weight': 3, 'lab validation accuracy': 0.996, 'final transport cost': 1.0914, 'feature size': 15.987, 'field entropy (info only)': 1.03, 'seconds': 5}
{'weight': 10, 'lab validation accuracy': 0.9915, 'final transport cost': 1.0636, 'feature size': 14.961, 'field entropy (info only)': 1.061, 'seconds': 5}


,lab validation accuracy,final transport cost,feature size,field entropy (info only),seconds
weight,,,,,
0.0,0.9977,NaN,51.718,0.359,4
0.1,0.9955,1.3550,26.954,0.995,4
0.3,0.9967,1.0251,12.082,0.907,12
1.0,0.9967,0.9421,13.286,0.862,8
3.0,0.9960,1.0914,15.987,1.030,5
10.0,0.9915,1.0636,14.961,1.061,5


In [9]:
reference = sweep.loc[0, "lab validation accuracy"]
reference_size = sweep.loc[0, "feature size"]

keeps_accuracy = [w for w in weight_grid if sweep.loc[w, "lab validation accuracy"] >= reference - TOLERANCE]
keeps_size = [w for w in weight_grid if sweep.loc[w, "feature size"] >= SCALE_FLOOR * reference_size]
allowed = [w for w in weight_grid if w in keeps_accuracy and w in keeps_size]
chosen_weight = max(allowed) if allowed else min(weight_grid)

print(f"Lab validation accuracy with no transport penalty: {reference:.4f} | feature size: {reference_size:.3f}")
print(f"Weights that keep the lab accuracy (within {TOLERANCE}): {keeps_accuracy}")
print(f"Weights that keep the features at least {SCALE_FLOOR:.0%} of their size:  {keeps_size}")
print(f"Chosen weight: {chosen_weight}", "" if allowed else "(none qualified, so we take the smallest)")
if chosen_weight == max(weight_grid):
    print("*** The chosen weight is the TOP of the grid: extend WEIGHT_GRID upwards and re-run Step 5. ***")

Lab validation accuracy with no transport penalty: 0.9977 | feature size: 51.718
Weights that keep the lab accuracy (within 0.01): [0.1, 0.3, 1, 3, 10]
Weights that keep the features at least 50% of their size:  [0.1]
Chosen weight: 0.1 


**Questions to think about:**
* Does the transport cost fall as the weight rises? If it doesn't, the penalty is too weak to matter.
* Look at the `field entropy` column. Does a stronger penalty make the network more or less decided about the field images?
  (Remember it is shown for information only. We never choose by it, and we never look at field labels here.)

## Step 6: All splits, then save

For each split we train with the chosen weight, using that split's calibration images as the unlabelled field batch. This is the
slowest notebook of the project, because every training step solves a transport problem. Expect several minutes per split.

In [10]:
def accuracy(logits, mask):
    return float((logits[mask].argmax(axis=1) == field_labels[mask]).mean())


save_dict = {"classes": np.array(classes), "align_weight": np.array(chosen_weight)}
for key in ["epochs_run", "debug_mode"]:
    if key in source.files:
        save_dict[key] = source[key]
for g in ["pv_val", "pv_test"]:
    save_dict[f"{g}_logits"] = source[f"{g}_logits"]
for g in ["pv_val", "pv_test", "pd_known", "pd_unknown"]:
    save_dict[f"{g}_labels"] = source[f"{g}_labels"]
    save_dict[f"{g}_paths"] = source[f"{g}_paths"]

for seed in seeds_to_run:
    start = time.time()
    is_cal = (split_info[f"B_seed{seed}"] == "cal").to_numpy()
    net, *_ = train_deepjdot(known_x[torch.from_numpy(is_cal).to(device)], seed, chosen_weight)

    save_dict[f"pd_known_logits_B_seed{seed}"] = logits_of(net, known_x)
    save_dict[f"pd_unknown_logits_B_seed{seed}"] = logits_of(net, unknown_x)
    print(f"split {seed}: {time.time() - start:.0f} s | test accuracy "
          f"{accuracy(save_dict[f'pd_known_logits_B_seed{seed}'], ~is_cal):.4f}")

suffix = "_QUICK" if QUICK_TEST else ""
output_file = OUT_DIR / f"outputs_{METHOD}_seed{TRAIN_SEED}{suffix}.npz"
np.savez_compressed(output_file, **save_dict)
print("Saved", output_file)

split 42: 5 s | test accuracy 0.2775
split 43: 9 s | test accuracy 0.2329
split 44: 6 s | test accuracy 0.2841
split 45: 8 s | test accuracy 0.2660
split 46: 6 s | test accuracy 0.2485
split 47: 6 s | test accuracy 0.2324
split 48: 11 s | test accuracy 0.2463
split 49: 5 s | test accuracy 0.2417
split 50: 8 s | test accuracy 0.2214
split 51: 10 s | test accuracy 0.2544
Saved outputs\outputs_deepjdot_seed42.npz


## Step 7: Evaluate with notebook 3b

Run once with `QUICK_TEST = True`, then set it to `False`, restart the kernel and run again. The real file is
`outputs_deepjdot_seed0.npz`. Add `"deepjdot"` to `METHODS` in notebook 3b.

**How to read DeepJDOT's rows.** Compare it with:
* **`bottleneck_only`**: the same network with no transport penalty. This is the fair control for what the penalty adds.
* **`jdot`**: optimal transport on fixed features. Does putting the transport inside the training help?
* **`deep_coral` and `jmmd`**: other ways of aligning the same network.
* **`adabn`**: the simplest method and so far the strongest.

This is the question of the whole study: do the more sophisticated methods (optimal transport and deeper feature
alignment) close the reliability gap, or do they share the fate of the lightweight ones?